# 13장. 외부 데이터로 분석을 확장하기

이 Notebook의 기본 실행은 **네트워크 요청 0회**입니다. 먼저 질문·출처·정책·저장 구조를 정의하고, 실제 수집은 현재 공식 문서와 이용조건을 사람이 확인한 뒤 `RUN_*` 플래그를 명시적으로 바꿨을 때만 수행합니다.

핵심 흐름: `Question → Official Source → Policy → RUN Gate → Raw → Metadata → Processed → Quality → Merge → Interpretation Limit`


## 학습 목표

- 공식 파일 → 공식 API → 제한적 공개 HTML 순서를 적용합니다.
- API Key 실제 값과 placeholder 상태를 구분하되 값 자체는 출력하지 않습니다.
- timeout·제한적 retry·pagination·응답 오류 구조를 공식 문서와 대조합니다.
- raw snapshot을 덮어쓰지 않고 metadata JSON과 processed 결과를 분리합니다.
- SHA-256을 원본 변경 탐지에 사용하되 데이터 타당성 보증으로 오해하지 않습니다.
- 외부 오른쪽 키 고유성, 병합 전후 행 수, left_only를 확인합니다.
- 검색 결과·동시 변화를 전체 여론·시장·인과관계로 과장하지 않습니다.


## 1. 프로젝트 루트와 공통 경로


In [ ]:
from pathlib import Path
import os
import sys

def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'requirements.txt').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('프로젝트 루트 폴더를 찾을 수 없습니다.')

PROJECT_ROOT = find_project_root(Path.cwd())
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('프로젝트 루트:', PROJECT_ROOT)
print('보고서 폴더:', REPORT_DIR)


## 2. 네트워크 없이 계획·폴더·Evidence 준비

이 단계에서는 외부 사이트에 연결하지 않습니다. 계획표, 수집 방법 비교, metadata 템플릿, API 코드 리뷰표, Network Gate를 먼저 생성합니다.


In [ ]:
from src.external_data_collection import run_external_data_collection_setup

setup_result = run_external_data_collection_setup(
    base_dir=PROJECT_ROOT,
    report_dir=REPORT_DIR,
)
outputs = setup_result['outputs']

display(outputs['data_plan'])
display(outputs['method_summary'])
display(outputs['integration_plan'])
display(outputs['network_gate'])


## 3. `.env` 인증정보 상태 — 값은 절대 출력하지 않기

`CONFIGURED / MISSING / PLACEHOLDER` 상태만 봅니다. `.env.example`을 그대로 복사해 `your_...` 값이 남아 있으면 `PLACEHOLDER`로 표시됩니다. 실제 Key 문자열, 인증 헤더, Secret 일부도 출력하지 않습니다.


In [ ]:
env_status = outputs['env_status']
display(env_status)
assert env_status['value_exposed'].eq(False).all()


## 4. HTTP Session — timeout과 제한적 retry

Requests의 `timeout`은 전체 다운로드 시간 제한과 동일한 개념이 아닙니다. 이 교재는 연결·읽기 timeout을 명시하고, GET 요청의 429·일부 일시적 5xx에만 제한적으로 재시도합니다. 400/401/403 같은 요청·인증 오류는 재시도로 해결하려 하지 않습니다.


In [ ]:
from src.external_data_collection import build_http_session

session = build_http_session(total_retries=3, backoff_factor=0.5)


## 5. 공식 JSON API — 기본 `RUN_PUBLIC_API=False`

아래 코드는 특정 공공 API의 주소나 parameter를 추측하지 않습니다. 선택한 데이터의 **현재 공식 문서**에서 URL, HTTP method, 인증 위치, parameter, response path, pagination, rate limit, 오류 구조를 확인한 뒤에만 값을 채웁니다.


In [ ]:
from src.external_data_collection import (
    build_collection_metadata,
    request_json_api,
    save_json_snapshot,
    save_metadata_snapshot,
    versioned_snapshot_path,
)

RUN_PUBLIC_API = False
PUBLIC_API_URL = ''  # 현재 공식 문서에서 확인한 실제 주소만 입력
PUBLIC_API_PARAMS = {
    # 공식 문서에서 확인한 parameter만 입력
}

if RUN_PUBLIC_API:
    if not PUBLIC_API_URL:
        raise ValueError('현재 공식 API URL을 먼저 확인하세요.')
    payload, request_meta = request_json_api(
        PUBLIC_API_URL,
        params=PUBLIC_API_PARAMS,
        session=session,
    )
    raw_path = versioned_snapshot_path(setup_result['paths']['raw'], 'public_api', '.json')
    save_json_snapshot(payload, raw_path)
    collection_meta = build_collection_metadata(
        provider='공식 제공기관',
        source_url=PUBLIC_API_URL,
        collection_method='official_api',
        data_reference_date='공식 문서에서 확인',
        request_scope='실제 요청 범위 기록',
        license_or_terms='현재 이용조건 확인 결과',
        raw_path=raw_path,
        policy_confirmed=True,
        extra=request_meta,
    )
    metadata_path = versioned_snapshot_path(setup_result['paths']['metadata'], 'public_api', '.json')
    save_metadata_snapshot(collection_meta, metadata_path)
else:
    print('RUN_PUBLIC_API=False — 네트워크 호출 없음')


## 6. 네이버 블로그 검색 API — 기본 `RUN_NAVER_API=False`

현재 공식 문서에서 Client ID/Secret은 HTTP header에 전달하며 `display`, `start`, `sort` 범위가 정의되어 있습니다. 이 값과 일일 호출 한도·정책은 실행 시점에 공식 개발자 문서에서 다시 확인합니다. 검색 결과는 전체 인터넷·전체 여론·시장 수요를 대표하지 않습니다.


In [ ]:
from src.external_data_collection import (
    NAVER_BLOG_DOCS_URL,
    build_collection_metadata,
    naver_blog_items_to_dataframe,
    save_json_snapshot,
    save_metadata_snapshot,
    search_naver_blog,
    validate_external_dataframe,
    versioned_snapshot_path,
)

RUN_NAVER_API = False

if RUN_NAVER_API:
    result, request_meta = search_naver_blog(
        '제주 여행', display=10, start=1, sort='sim', session=session
    )
    raw_path = versioned_snapshot_path(setup_result['paths']['raw'], 'naver_blog_jeju', '.json')
    save_json_snapshot(result, raw_path)

    naver_df = naver_blog_items_to_dataframe(result)
    quality = validate_external_dataframe(naver_df, key_columns='link', date_columns='postdate')
    display(quality)
    if quality['status'].eq('FAIL').any():
        raise ValueError('네이버 검색 결과 품질 검증에 실패했습니다.')

    processed_path = versioned_snapshot_path(setup_result['paths']['processed'], 'naver_blog_jeju', '.csv')
    naver_df.to_csv(processed_path, index=False, encoding='utf-8-sig')
    collection_meta = build_collection_metadata(
        provider='Naver Search API',
        source_url=NAVER_BLOG_DOCS_URL,
        collection_method='official_api',
        data_reference_date='검색 결과의 postdate 및 수집 시각 별도 기록',
        request_scope='query=제주 여행, display=10, start=1, sort=sim',
        license_or_terms='현재 Naver 개발자 정책 재확인',
        raw_path=raw_path,
        processed_path=processed_path,
        policy_confirmed=True,
        extra=request_meta,
    )
    metadata_path = versioned_snapshot_path(setup_result['paths']['metadata'], 'naver_blog_jeju', '.json')
    save_metadata_snapshot(collection_meta, metadata_path)
else:
    print('RUN_NAVER_API=False — 네트워크 호출 없음')


## 7. 제한적 공개 HTML — `RUN_CRAWLING_EXAMPLE=False`, `POLICY_CONFIRMED=False`

공식 파일/API가 없는지 먼저 확인합니다. robots.txt는 자동 접근 규칙이지 계약상 이용허락이나 저작권 허가가 아닙니다. 로그인·CAPTCHA·paywall·접근제한을 우회하지 않습니다.


In [ ]:
from src.external_data_collection import (
    build_collection_metadata,
    extract_title_and_links,
    fetch_public_html,
    save_metadata_snapshot,
    save_text_snapshot,
    versioned_snapshot_path,
)

RUN_CRAWLING_EXAMPLE = False
POLICY_CONFIRMED = False
TARGET_URL = ''  # 실제 허용된 공개 페이지를 정책 확인 후 입력

if RUN_CRAWLING_EXAMPLE and POLICY_CONFIRMED:
    if not TARGET_URL:
        raise ValueError('검토한 공개 URL을 입력하세요.')
    html, request_meta = fetch_public_html(
        TARGET_URL, policy_confirmed=True, respect_robots=True, session=session
    )
    raw_path = versioned_snapshot_path(setup_result['paths']['raw'], 'public_page', '.html')
    save_text_snapshot(html, raw_path)
    page_title, links_df = extract_title_and_links(html, base_url=TARGET_URL)
    processed_path = versioned_snapshot_path(setup_result['paths']['processed'], 'public_links', '.csv')
    links_df.to_csv(processed_path, index=False, encoding='utf-8-sig')
    collection_meta = build_collection_metadata(
        provider='웹사이트 운영 주체',
        source_url=TARGET_URL,
        collection_method='limited_public_html',
        data_reference_date='페이지 기준일 확인',
        request_scope='승인된 단일/소량 페이지',
        license_or_terms='이용약관·저작권·개인정보 검토 결과',
        raw_path=raw_path, processed_path=processed_path, policy_confirmed=True, extra=request_meta,
    )
    metadata_path = versioned_snapshot_path(setup_result['paths']['metadata'], 'public_page', '.json')
    save_metadata_snapshot(collection_meta, metadata_path)
    print('페이지 제목:', page_title)
else:
    print('크롤링 비활성화 — 정책·robots.txt·이용조건 확인 전 실행하지 않음')


## 8. 외부 데이터 품질과 병합 검증

아래 DataFrame은 **병합 규칙을 설명하기 위한 합성 예제**이며 실제 공휴일 분석 결과가 아닙니다. 실제 프로젝트에서는 출처·기준일·metadata가 있는 외부 processed 파일만 사용합니다.


In [ ]:
import pandas as pd
from src.external_data_collection import merge_external_data, validate_external_dataframe

monthly_sales_demo = pd.DataFrame({
    'order_month': ['2026-01', '2026-02', '2026-03'],
    'completed_order_amount': [1200000, 1500000, 1300000],
})
external_monthly_demo = pd.DataFrame({
    'order_month': ['2026-01', '2026-02', '2026-03'],
    'external_indicator': [3, 1, 2],
})

quality = validate_external_dataframe(external_monthly_demo, key_columns='order_month')
display(quality)
if quality['status'].eq('FAIL').any():
    raise ValueError('외부 데이터 key 품질을 먼저 수정하세요.')

merged_demo, merge_check = merge_external_data(
    monthly_sales_demo, external_monthly_demo, on='order_month', how='left', validate='many_to_one'
)
display(merged_demo)
display(merge_check)


`left_only_count > 0`은 곧바로 데이터 오류라는 뜻은 아니지만 외부 데이터의 기간·키 정의가 내부 분석 범위와 맞는지 확인해야 하는 WARN입니다. 함께 움직이는 두 지표를 원인과 결과로 단정하지 않습니다.


## 9. LLM이 만든 API 코드 검토

LLM에는 실제 API Key·Secret·내부 URL·개인정보를 주지 않습니다. 공식 문서에서 사람이 확인한 URL 형식, method, parameter, response path, rate limit만 제공하고 아래 체크리스트로 다시 검토합니다.


In [ ]:
display(outputs['api_code_review'])
display(outputs['metadata_template'])
display(outputs['external_data_log'])


## 10. 생성된 준비 결과 확인


In [ ]:
for name, path in setup_result['output_paths'].items():
    print(name, 'OK' if path.exists() else 'MISSING', path)


## 11. 네트워크 없는 Setup 재실행

프로젝트 루트에서 다음 명령을 실행하면 네트워크 요청 없이 계획·Gate·검토 템플릿을 다시 생성합니다.

```powershell
python scripts/run_external_data_collection.py
```


## 정리

외부 데이터 수집에서 중요한 것은 요청 코드를 빨리 만드는 것이 아니라 **질문에 필요한 최소 데이터인지, 현재 정책상 수집 가능한지, Secret이 보호되는지, raw와 metadata가 남는지, 내부 데이터와 같은 단위로 안전하게 연결되는지**를 증명하는 것입니다.
